# 03 — IA aplicada: classificação de risco de fogo

**Curso:** Tecnologia, Dados e IA na Gestão Ambiental
**Encontros:** 6 e 9

Neste notebook você vai treinar um modelo simples de **machine learning supervisionado** para
prever risco de foco de calor a partir de variáveis climáticas. O objetivo não é competir por
acurácia: é **entender o erro e documentar o modelo**.

> O arquivo `exemplo-risco-fogo-sintetico.csv` é didático. Para dados reais, use INMET (clima) e
> INPE (focos de calor).

## Passo 1 — Importar

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

print("bibliotecas carregadas")

## Passo 2 — Ler os dados

In [ ]:
df = pd.read_csv("exemplo-risco-fogo-sintetico.csv")
df.head()

In [ ]:
df["risco_fogo"].value_counts(normalize=True).rename("proporcao")

## Passo 3 — Separar treino e teste

Nunca avalie o modelo com os mesmos dados que ele usou para aprender. Aqui, 70% para treino e
30% para teste.

In [ ]:
entradas = ["temp_max_c", "umidade_rel_pct", "vento_kmh", "chuva_mm"]
X = df[entradas]
y = df["risco_fogo"]

X_treino, X_teste, y_treino, y_teste = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

print("treino:", X_treino.shape, "teste:", X_teste.shape)

## Passo 4 — Treinar a árvore de decisão

In [ ]:
modelo = DecisionTreeClassifier(max_depth=3, random_state=42)
modelo.fit(X_treino, y_treino)
print("modelo treinado")

## Passo 5 — Avaliar

In [ ]:
y_pred = modelo.predict(X_teste)

print("Acuracia:", round(accuracy_score(y_teste, y_pred), 3))
print()
print("Matriz de confusao:")
print(confusion_matrix(y_teste, y_pred))
print()
print(classification_report(y_teste, y_pred, digits=3))

Leia a matriz de confusão:

- **falso positivo:** o modelo disse "risco" e não era;
- **falso negativo:** o modelo disse "sem risco" e era risco.

Na gestão ambiental, os dois erros têm custos diferentes. Qual é o pior no seu contexto?

## Passo 6 — Ver as regras aprendidas

In [ ]:
print(export_text(modelo, feature_names=entradas))

## Passo 7 — Importância das variáveis

In [ ]:
importancia = pd.Series(modelo.feature_importances_, index=entradas).sort_values()
importancia.plot(kind="barh", figsize=(7, 3), title="Importancia das variaveis")
plt.tight_layout()
plt.show()

## Exercício — variar a profundidade

In [ ]:
resultados = []
for profundidade in range(1, 9):
    m = DecisionTreeClassifier(max_depth=profundidade, random_state=42)
    m.fit(X_treino, y_treino)
    acc = accuracy_score(y_teste, m.predict(X_teste))
    resultados.append((profundidade, round(acc, 3)))

pd.DataFrame(resultados, columns=["max_depth", "acuracia"])

## Ficha de modelo (preencha)

- Nome do modelo:
- O que ele faz:
- Dados de treino (fonte, período, tamanho):
- Variáveis de entrada:
- Rótulo previsto:
- Métrica e resultado:
- Falsos positivos / falsos negativos:
- Onde ele erra mais:
- Quem é afetado pelo erro:
- Limitação principal:
- Uso recomendado / não recomendado:

## Uso de IA com verificação

Prompt sugerido:

> "Treinei uma árvore de decisão com as variáveis [liste] para prever risco de fogo. A acurácia
> no teste foi [valor] e a matriz de confusão foi [valores]. Explique em linguagem simples o que
> isso significa e liste 3 limitações. Não invente números."

Confira cada afirmação com a saída do notebook e registre o prompt.

## Discussão ética

1. Se o modelo errar um falso negativo, quem pode ser prejudicado?
2. Se os dados de treino vierem só de uma região, o modelo funciona em outra?
3. Quem decide o que é "risco"? O modelo ou a gestão pública?
4. O que precisa estar escrito na ficha para o modelo ser auditável?